In [2]:
import pandas as pd

In [3]:
file_path = 'Data_AAPL.csv' 
df = pd.read_csv(file_path)

C:\Users\oopas\AppData\Local\Temp\ipykernel_6868\460625098.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [4]:
df.columns = df.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.strip()
print("Очищені назви колонок:", df.columns.tolist())

Очищені назви колонок: ['QUOTE_READTIME', 'QUOTE_DATE', 'UNDERLYING_LAST', 'EXPIRE_DATE', 'DTE', 'C_LAST', 'STRIKE', 'log_ret', 'historical_vol', 'vol_30d', 'vol_garch']


In [5]:
df['C_LAST'] = pd.to_numeric(df['C_LAST'], errors='coerce')
df['C_LAST'] = df['C_LAST'].fillna(0.0)
df_clean = df[df['C_LAST'] > 0.001].copy()

df_clean['QUOTE_DATE'] = pd.to_datetime(df_clean['QUOTE_DATE'])
df_clean['EXPIRE_DATE'] = pd.to_datetime(df_clean['EXPIRE_DATE'])

In [6]:
df_clean['Quote_Month'] = df_clean['QUOTE_DATE'].dt.to_period('M')

In [7]:
def get_most_popular(x):
    mode = x.mode()
    if not mode.empty:
        return mode.iloc[0] 
    return None

stats = df_clean.groupby(['Quote_Month', 'DTE'])['STRIKE'].agg(
    Most_Popular_Strike=get_most_popular, 
    Average_Strike='mean',                
    Count='count'                         
).reset_index()

stats.head()

,Quote_Month,DTE,Most_Popular_Strike,Average_Strike,Count
0,2020-09,0.0,57.5,110.762506,423
1,2020-09,1.0,57.5,111.526538,416
2,2020-09,2.0,60.0,113.081242,499
3,2020-09,3.0,57.5,111.530894,414
4,2020-09,4.0,60.0,113.360129,311


In [8]:
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd

stats = stats.sort_values(by=['Quote_Month', 'DTE'])

unique_months = stats['Quote_Month'].astype(str).unique()

for month in unique_months:
    df_plot = stats[stats['Quote_Month'].astype(str) == month]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_plot['DTE'], 
        y=df_plot['Most_Popular_Strike'],
        mode='lines+markers', 
        name='Most Popular (Mode)',
        line=dict(color='blue', width=2),
        marker=dict(size=6),
        hovertemplate='<b>DTE: %{x}</b><br>Strike: %{y}<br>Type: Most Popular<extra></extra>' 
    ))

    fig.add_trace(go.Scatter(
        x=df_plot['DTE'], 
        y=df_plot['Average_Strike'],
        mode='lines', 
        name='Average (Mean)',
        line=dict(color='orange', width=2, dash='dash'), 
        hovertemplate='<b>DTE: %{x}</b><br>Avg Strike: %{y:.2f}<br>Type: Average<extra></extra>'
    ))

    fig.update_layout(
        title=f'Strike Analysis: {month}',
        xaxis_title='DTE (Days to Expiration)',
        yaxis_title='Strike Price',
        template='plotly_white',
        height=600, 
        hovermode="x unified", 
        
        xaxis=dict(
            rangeslider=dict(visible=True), 
            type="linear"
        )
    )
    
    pio.renderers.default = "notebook_connected"
    
    fig.show()

In [9]:
df_clean.to_csv('data_appl_cleaned.csv', index=False)

In [10]:
df_clean['diff_pct'] = abs(df_clean['UNDERLYING_LAST'] - df_clean['STRIKE']) / df_clean['UNDERLYING_LAST']

threshold = 0.01 

atm_calls = df_clean[df_clean['diff_pct'] <= threshold].copy()

atm_calls_sorted = atm_calls.sort_values(by=['QUOTE_DATE', 'diff_pct'])

cols_to_show = ['QUOTE_DATE', 'UNDERLYING_LAST', 'STRIKE', 'C_LAST', 'DTE', 'diff_pct']
print(f"Знайдено {len(atm_calls_sorted)} ATM опціонів.")
atm_calls_sorted[cols_to_show].head(20)

Знайдено 17247 ATM опціонів.


,QUOTE_DATE,UNDERLYING_LAST,STRIKE,C_LAST,DTE,diff_pct
125,2020-09-01,134.22,133.75,18.92,227.00,0.003502
244,2020-09-01,134.22,133.75,13.17,80.04,0.003502
859,2020-09-01,134.22,133.75,6.10,17.00,0.003502
928,2020-09-01,134.22,133.75,2.83,3.00,0.003502
1017,2020-09-01,134.22,133.75,4.65,10.00,0.003502
1109,2020-09-01,134.22,133.75,9.74,45.00,0.003502
1335,2020-09-01,134.22,133.75,7.35,24.00,0.003502
1386,2020-09-01,134.22,133.75,9.00,38.00,0.003502
1422,2020-09-01,134.22,133.75,8.25,31.00,0.003502
72,2020-09-01,134.22,135.00,14.95,136.04,0.005811


In [11]:
atm_calls_sorted = atm_calls_sorted.sort_values(by=['QUOTE_DATE', 'DTE', 'diff_pct'])

best_per_dte = atm_calls_sorted.drop_duplicates(subset=['QUOTE_DATE', 'DTE'], keep='first')

cols_to_show = ['QUOTE_DATE', 'UNDERLYING_LAST', 'STRIKE', 'C_LAST', 'DTE', 'diff_pct']
print(f"Відібрано {len(best_per_dte)} унікальних DTE (найкращий страйк для кожного).")
print(best_per_dte[cols_to_show].to_string(index=False))

Відібрано 12017 унікальних DTE (найкращий страйк для кожного).
QUOTE_DATE  UNDERLYING_LAST  STRIKE  C_LAST     DTE  diff_pct
2020-09-01           134.22  133.75    2.83    3.00  0.003502
2020-09-01           134.22  133.75    4.65   10.00  0.003502
2020-09-01           134.22  133.75    6.10   17.00  0.003502
2020-09-01           134.22  133.75    7.35   24.00  0.003502
2020-09-01           134.22  133.75    8.25   31.00  0.003502
2020-09-01           134.22  133.75    9.00   38.00  0.003502
2020-09-01           134.22  133.75    9.74   45.00  0.003502
2020-09-01           134.22  133.75   13.17   80.04  0.003502
2020-09-01           134.22  135.00   13.59  108.04  0.005811
2020-09-01           134.22  135.00   14.95  136.04  0.005811
2020-09-01           134.22  135.00   17.52  199.00  0.005811
2020-09-01           134.22  133.75   18.92  227.00  0.003502
2020-09-01           134.22  135.00   20.53  290.00  0.005811
2020-09-01           134.22  135.00   22.68  381.00  0.005811
2020-09

In [ ]:
atm_calls_sorted['EXP_MONTH_STR'] = atm_calls_sorted['EXPIRE_DATE'].dt.to_period('M')

all_months = atm_calls_sorted['EXP_MONTH_STR'].unique()
total_months_count = len(all_months)
print(f"Всього унікальних місяців у даних: {total_months_count}")

strike_counts = atm_calls_sorted.groupby('STRIKE')['EXP_MONTH_STR'].nunique()

common_strikes = strike_counts[strike_counts == total_months_count]

if len(common_strikes) > 0:
    print(f"\nЗнайдено страйки, що є в кожному місяці: {list(common_strikes.index)}")
    
    target_strike = common_strikes.index[0] # беремо перший знайдений
    print(f"\nПриклад для страйку {target_strike}:")
    
    subset = atm_calls_sorted[atm_calls_sorted['STRIKE'] == target_strike]
    subset = subset.sort_values('EXPIRE_DATE')
    print(subset[['EXPIRE_DATE', 'STRIKE', 'C_LAST', 'DTE']].drop_duplicates(subset=['EXPIRE_DATE']).to_string(index=False))
else:
    print("\nНа жаль, жоден страйк не зустрічається абсолютно в усіх місяцях.")
    print("Найпопулярніші страйки (страйк: кількість місяців):")
    print(strike_counts.sort_values(ascending=False).head(3).to_string())

Всього унікальних місяців у даних: 54

На жаль, жоден страйк не зустрічається абсолютно в усіх місяцях.
Найпопулярніші страйки (страйк: кількість місяців):
STRIKE
135.0    40
140.0    40
130.0    40


In [ ]:
atm_calls_sorted['QUOTE_MONTH_STR'] = atm_calls_sorted['QUOTE_DATE'].dt.to_period('M')

all_months = atm_calls_sorted['QUOTE_MONTH_STR'].unique()
total_months_count = len(all_months)
print(f"Всього унікальних місяців у даних: {total_months_count}")

strike_counts = atm_calls_sorted.groupby('STRIKE')['QUOTE_MONTH_STR'].nunique()

common_strikes = strike_counts[strike_counts == total_months_count]

if len(common_strikes) > 0:
    print(f"\nЗнайдено страйки, що є в кожному місяці: {list(common_strikes.index)}")
    
    target_strike = common_strikes.index[0] 
    print(f"\nПриклад для страйку {target_strike}:")
    
    subset = atm_calls_sorted[atm_calls_sorted['STRIKE'] == target_strike]
    subset = subset.sort_values('QUOTE_DATE')
    print(subset[['QUOTE_DATE', 'STRIKE', 'C_LAST', 'DTE']].drop_duplicates(subset=['QUOTE_DATE']).to_string(index=False))
else:
    print("\nНа жаль, жоден страйк не зустрічається абсолютно в усіх місяцях.")
    print("Найпопулярніші страйки (страйк: кількість місяців):")
    print(strike_counts.sort_values(ascending=False).head(3).to_string())

Всього унікальних місяців у даних: 40

На жаль, жоден страйк не зустрічається абсолютно в усіх місяцях.
Найпопулярніші страйки (страйк: кількість місяців):
STRIKE
147.0    14
149.0    14
150.0    14


In [ ]:
atm_calls_sorted['EXPIRE_DATE'] = pd.to_datetime(atm_calls_sorted['EXPIRE_DATE'])

atm_calls_sorted['YEAR'] = atm_calls_sorted['EXPIRE_DATE'].dt.year
atm_calls_sorted['MONTH_STR'] = atm_calls_sorted['EXPIRE_DATE'].dt.strftime('%m') 

strike_counts = atm_calls_sorted['STRIKE'].value_counts()
popular_strikes = strike_counts[strike_counts >= 10].index

df_filtered = atm_calls_sorted[atm_calls_sorted['STRIKE'].isin(popular_strikes)]

grouped = df_filtered.groupby(['STRIKE', 'YEAR'])['MONTH_STR'].unique()

print(f"{'СТРАЙК':<10} | {'РІК':<6} | {'К-ТЬ':<5} | {'МІСЯЦІ ЕКСПІРАЦІЇ'}")
print("-" * 65)

for (strike, year), months in grouped.items():
    sorted_months = sorted(months)
    months_str = ", ".join(sorted_months)
    count = len(sorted_months)
    
    marker = "✅" if count >= 11 else "⚠️" if count < 6 else " "
    
    if count >= 3:
        print(f"{strike:<10} | {year:<6} | {count:<5} | {marker} {months_str}")

print("-" * 65)
print("✅ - рік майже повний, можна брати для тесту.")

СТРАЙК     | РІК    | К-ТЬ  | МІСЯЦІ ЕКСПІРАЦІЇ
-----------------------------------------------------------------
106.25     | 2020   | 3     | ⚠️ 09, 10, 11
107.5      | 2020   | 4     | ⚠️ 09, 10, 11, 12
107.5      | 2021   | 5     | ⚠️ 01, 03, 04, 06, 09
107.5      | 2022   | 3     | ⚠️ 01, 06, 09
108.0      | 2020   | 3     | ⚠️ 10, 11, 12
108.75     | 2020   | 3     | ⚠️ 09, 10, 11
109.0      | 2020   | 3     | ⚠️ 10, 11, 12
110.0      | 2020   | 4     | ⚠️ 09, 10, 11, 12
110.0      | 2021   | 6     |   01, 02, 03, 04, 06, 09
110.0      | 2022   | 3     | ⚠️ 01, 06, 09
111.0      | 2020   | 3     | ⚠️ 10, 11, 12
111.25     | 2020   | 3     | ⚠️ 09, 10, 11
112.0      | 2020   | 3     | ⚠️ 10, 11, 12
112.5      | 2020   | 4     | ⚠️ 09, 10, 11, 12
112.5      | 2021   | 5     | ⚠️ 01, 03, 04, 06, 09
112.5      | 2022   | 3     | ⚠️ 01, 06, 09
113.0      | 2020   | 3     | ⚠️ 10, 11, 12
113.75     | 2020   | 3     | ⚠️ 09, 10, 11
114.0      | 2020   | 3     | ⚠️ 10, 11, 12
115.0      

In [16]:
atm_calls_sorted.to_csv('atm_calls_sorted.csv', index=False)